In [1]:
# hit_generator.py
# Single-voice drum step generator (16th-note grid) with continuous velocity
# IMPORTANT: training data is loaded as (file, pitch) records via MultiVoiceMidiDataset
# so each sample is ONE pitch lane, not a whole multi-pitch MIDI file.

from __future__ import annotations

import subprocess
import signal
import sys
import os
import random
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, Tuple, Optional, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from mid_to_velocity import MidiGridConfig, MultiVoiceMidiDataset
from midi_export import export_single_voice_steps_to_midi, MidiExportConfig


# ----------------------------
# Utils
# ----------------------------

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def circular_pad_1d(x: torch.Tensor, pad: int) -> torch.Tensor:
    """
    Circular padding for 1D conv.
    x: (B, C, T)
    """
    if pad <= 0:
        return x
    return torch.cat([x[..., -pad:], x, x[..., :pad]], dim=-1)


def masked_huber(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor, delta: float = 0.1) -> torch.Tensor:
    """
    pred/target: (B, T)
    mask: (B, T) in {0,1} or [0,1]
    """
    loss = F.smooth_l1_loss(pred, target, reduction="none", beta=delta)  # (B, T)
    denom = mask.sum().clamp_min(1.0)
    return (loss * mask).sum() / denom


def _ensure_pt_name(name: str) -> str:
    return name if name.endswith(".pt") else (name + ".pt")


def make_unique_model_name(prefix: str = "single_voice_tcn") -> str:
    # Example: single_voice_tcn_20260226_153012_482931
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    suffix = f"{random.randint(0, 999999):06d}"
    return f"{prefix}_{ts}_{suffix}"


# ----------------------------
# Model: TCN with circular padding
# ----------------------------

class CircularConv1d(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int, dilation: int):
        super().__init__()
        assert kernel_size % 2 == 1, "Use odd kernel size for 'same' length."
        self.kernel_size = kernel_size
        self.dilation = dilation
        self.conv = nn.Conv1d(
            in_channels=in_ch,
            out_channels=out_ch,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=0,  # manual circular padding
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pad = self.dilation * (self.kernel_size - 1) // 2
        x = circular_pad_1d(x, pad)
        return self.conv(x)


class TCNBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int, dilation: int, dropout: float = 0.1):
        super().__init__()
        self.conv1 = CircularConv1d(channels, channels, kernel_size=kernel_size, dilation=dilation)
        self.conv2 = CircularConv1d(channels, channels, kernel_size=kernel_size, dilation=dilation)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.GroupNorm(num_groups=8 if channels >= 8 else 1, num_channels=channels)
        self.norm2 = nn.GroupNorm(num_groups=8 if channels >= 8 else 1, num_channels=channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x

        x = self.conv1(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.dropout(x)

        x = self.conv2(x)
        x = self.norm2(x)
        x = F.gelu(x)
        x = self.dropout(x)

        return x + residual


class SingleVoiceTCN(nn.Module):
    """
    Input per step:
      - hit_in (0/1, possibly corrupted)
      - vel_in (0..1, possibly corrupted)
      - mask_in (1 if masked/unknown, 0 if known)
    x: (B, 3, T)

    Output (SINGLE PITCH GRID):
      - hit_logits: (B, T)
      - vel_mu: (B, T) in [0,1]
      - vel_log_sigma: (B, T) optional
    """

    def __init__(
        self,
        in_channels: int = 3,
        hidden: int = 64,
        kernel_size: int = 3,
        dilations: Tuple[int, ...] = (1, 2, 4, 8, 16, 1, 2, 4),
        dropout: float = 0.1,
        predict_sigma: bool = True,
    ):
        super().__init__()
        self.predict_sigma = predict_sigma

        self.in_proj = nn.Conv1d(in_channels, hidden, kernel_size=1)
        self.blocks = nn.ModuleList([TCNBlock(hidden, kernel_size=kernel_size, dilation=d, dropout=dropout) for d in dilations])

        self.hit_head = nn.Conv1d(hidden, 1, kernel_size=1)  # logits
        self.vel_head = nn.Conv1d(hidden, 1, kernel_size=1)  # mu -> sigmoid

        self.sigma_head = nn.Conv1d(hidden, 1, kernel_size=1) if predict_sigma else None

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        h = self.in_proj(x)
        for blk in self.blocks:
            h = blk(h)

        hit_logits = self.hit_head(h).squeeze(1)                 # (B, T)
        vel_mu = torch.sigmoid(self.vel_head(h).squeeze(1))      # (B, T)

        out = {"hit_logits": hit_logits, "vel_mu": vel_mu}
        if self.predict_sigma and self.sigma_head is not None:
            out["vel_log_sigma"] = self.sigma_head(h).squeeze(1) # (B, T)

        return out


# ----------------------------
# Masking / corruption for denoising
# ----------------------------

@dataclass
class MaskConfig:
    p_mask: float = 0.5
    p_flip_hit: float = 0.0
    p_jitter_vel: float = 0.0


def corrupt_inputs(
    hit: torch.Tensor,  # (B,T)
    vel: torch.Tensor,  # (B,T)
    cfg: MaskConfig,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Returns: hit_in, vel_in, mask (1 where masked)
    """
    B, T = hit.shape
    device = hit.device

    mask = (torch.rand((B, T), device=device) < cfg.p_mask).float()

    hit_in = hit.clone()
    vel_in = vel.clone()

    # masked steps -> 0, model uses mask channel to know it's unknown
    hit_in = hit_in * (1.0 - mask)
    vel_in = vel_in * (1.0 - mask)

    # optional corruption on known steps
    if cfg.p_flip_hit > 0.0:
        flip = ((torch.rand((B, T), device=device) < cfg.p_flip_hit).float()) * (1.0 - mask)
        hit_in = torch.where(flip > 0.0, 1.0 - hit_in, hit_in)

    if cfg.p_jitter_vel > 0.0:
        noise = torch.randn((B, T), device=device) * cfg.p_jitter_vel
        vel_in = torch.clamp(vel_in + noise * (1.0 - mask), 0.0, 1.0)

    return hit_in, vel_in, mask


# ----------------------------
# DATA LOADING (FIXED): per-pitch samples, not per-file
# ----------------------------

def load_training_tensors(
    midi_dir: str,
    cfg: MidiGridConfig,
    *,
    batch_size: int = 256,
    min_hits_per_voice: int = 1,
    max_hits_per_voice: Optional[int] = None,
    device: str = "cpu",
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Returns:
      data_hit: (N, T)
      data_vel: (N, T)

    N counts *pitches*, not files:
    MultiVoiceMidiDataset indexes (file, pitch) pairs and each item returns
    one pitch-lane grid. (This is the core fix.)
    """
    ds = MultiVoiceMidiDataset(
        midi_dir,
        cfg,
        min_hits_per_voice=min_hits_per_voice,
        max_hits_per_voice=max_hits_per_voice,
        include_pitch=False,  # pitch not needed for training
    )

    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False)

    all_hit: List[torch.Tensor] = []
    all_vel: List[torch.Tensor] = []

    for batch in dl:
        # batch["hit"], batch["vel"] : (B, T)
        all_hit.append(batch["hit"])
        all_vel.append(batch["vel"])

    data_hit = torch.cat(all_hit, dim=0).float().to(device)
    data_vel = torch.cat(all_vel, dim=0).float().to(device)

    return data_hit, data_vel


# ----------------------------
# Saving / Loading
# ----------------------------

def save_model(
    model: nn.Module,
    path: str,
    *,
    model_config: Optional[dict] = None,
    best_loss: Optional[float] = None,
) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_config": model_config,
            "best_loss": best_loss,
        },
        path,
    )


def load_checkpoint(path: str, device: str = "cpu") -> dict:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    return torch.load(path, map_location=torch.device(device), weights_only = True)


# ----------------------------
# Training
# ----------------------------

def train(
    model: nn.Module,
    data_hit: torch.Tensor,   # (N, T)
    data_vel: torch.Tensor,   # (N, T)
    *,
    epochs: Optional[int] = 20,
    batch_size: int = 128,
    lr: float = 2e-3,
    device: str = "cpu",
    mask_cfg: MaskConfig = MaskConfig(p_mask=0.5),
    lambda_vel: float = 1.0,
    save_best_path: Optional[str] = None,
    model_config: Optional[dict] = None,
    best_init: float = float("inf"),
) -> float:
    device_t = torch.device(device)
    model.to(device_t)

    # move data if needed
    data_hit = data_hit.to(device_t)
    data_vel = data_vel.to(device_t)

    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    best = float(best_init)

    N, T = data_hit.shape

    # simple minibatching from tensors
    def batches():
        idx = torch.randperm(N, device=device_t)
        for i in range(0, N, batch_size):
            j = idx[i:i + batch_size]
            yield data_hit[j], data_vel[j]

    epoch = 0
    while True:
        if epochs is not None and epoch >= epochs:
            break

        model.train()
        running = 0.0
        n_seen = 0

        for hit, vel in batches():
            hit_in, vel_in, mask = corrupt_inputs(hit, vel, mask_cfg)

            x = torch.stack([hit_in, vel_in, mask], dim=1)  # (B, 3, T)
            out = model(x)

            # hit loss on masked steps only
            hit_loss = F.binary_cross_entropy_with_logits(out["hit_logits"], hit, reduction="none")
            hit_loss = (hit_loss * mask).sum() / mask.sum().clamp_min(1.0)

            # velocity loss only on masked steps (and only where there is a hit)
            vel_mask = mask * hit  # only care vel where a hit exists
            vel_loss = masked_huber(out["vel_mu"], vel, vel_mask, delta=0.1)

            loss = hit_loss + lambda_vel * vel_loss

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            running += float(loss.item()) * hit.shape[0]
            n_seen += hit.shape[0]

        epoch_loss = running / max(1, n_seen)
        print(f"epoch {epoch}  loss={epoch_loss:.6f}")

        if save_best_path is not None and epoch_loss < best:
            best = epoch_loss
            save_model(model, path=save_best_path, model_config=model_config, best_loss=best)
            print(f"  saved best -> {save_best_path} (best={best:.6f})")

        epoch += 1

    return best


# ----------------------------
# Example main
# ----------------------------

if __name__ == "__main__":
    set_seed(42)

    cfg = MidiGridConfig(
        bars=8,
        steps_per_bar=16,
        drum_channel=None,
        voice_pitch=None,          # ignored by MultiVoiceMidiDataset indexing
        step_merge="max",
        quantize="nearest",
        align_first_hit_to_zero=False,
    )

    device = "cpu"
    data_hit, data_vel = load_training_tensors(
        midi_dir="MIDI",
        cfg=cfg,
        batch_size=256,
        min_hits_per_voice=4,
        max_hits_per_voice=64,
        device=device,
    )
    N, T = data_hit.shape
    print("Loaded per-pitch samples:", (N, T))

    # Pass a filename (e.g. "candidate.pt" or "candidate") to resume.
    # Pass None to start a new run with a unique filename.
    resume_name: Optional[str] = "single_voice_tcn"  # e.g. "single_voice_tcn_20260226_153012_482931" or "candidate.pt"

    default_model_cfg = {
        "in_channels": 3,
        "hidden": 64,
        "kernel_size": 3,
        "dilations": (1, 2, 4, 8, 16, 1, 2, 4),
        "dropout": 0.1,
        "predict_sigma": True,
    }

    best_init = float("inf")

    if resume_name is not None:
        ckpt_path = os.path.join("Models", _ensure_pt_name(resume_name))
        ckpt = load_checkpoint(ckpt_path, device=device)
        model_cfg = ckpt.get("model_config") or default_model_cfg
        model = SingleVoiceTCN(**model_cfg)
        model.load_state_dict(ckpt["model_state_dict"])
        best_init = float(ckpt.get("best_loss", float("inf")))
        model_name = os.path.splitext(os.path.basename(ckpt_path))[0]
        best_path = ckpt_path
        print(f"Resuming from: {best_path} (prev_best={best_init if best_init < float('inf') else 'unknown'})")
    else:
        model_cfg = default_model_cfg
        model = SingleVoiceTCN(**model_cfg)
        model_name = make_unique_model_name(prefix="single_voice_tcn")
        best_path = os.path.join("Models", _ensure_pt_name(model_name))
        print(f"Starting new run: {best_path}")

    caff = None
    
    try:
        caff = subprocess.Popen(["caffeinate", "-dimsu"])
    
        train(
            model=model,
            data_hit=data_hit,
            data_vel=data_vel,
            epochs=None,
            batch_size=128,
            lr=2e-3,
            device=device,
            mask_cfg=MaskConfig(p_mask=0.55),
            lambda_vel=1.5,
            save_best_path=best_path,
            model_config=model_cfg,
            best_init=best_init,
        )
        
    except KeyboardInterrupt:
        print("\nStopped by user. Done.")
    finally:
        if caff is not None:
            caff.terminate()
            caff.wait()

Loaded per-pitch samples: (663, 128)
Resuming from: Models/single_voice_tcn.pt (prev_best=0.19260845084985098)
epoch 0  loss=0.267202
epoch 1  loss=0.251189
epoch 2  loss=0.241376
epoch 3  loss=0.242791
epoch 4  loss=0.241015
epoch 5  loss=0.237790
epoch 6  loss=0.232506
epoch 7  loss=0.235014
epoch 8  loss=0.233486
epoch 9  loss=0.233773
epoch 10  loss=0.232508
epoch 11  loss=0.229652
epoch 12  loss=0.229773
epoch 13  loss=0.230944
epoch 14  loss=0.229015
epoch 15  loss=0.233582
epoch 16  loss=0.231043
epoch 17  loss=0.231882
epoch 18  loss=0.233021
epoch 19  loss=0.229414
epoch 20  loss=0.228682
epoch 21  loss=0.225935
epoch 22  loss=0.233227
epoch 23  loss=0.227742
epoch 24  loss=0.228688
epoch 25  loss=0.224911
epoch 26  loss=0.225617
epoch 27  loss=0.223995
epoch 28  loss=0.230981
epoch 29  loss=0.231266
epoch 30  loss=0.227026
epoch 31  loss=0.232262
epoch 32  loss=0.223971
epoch 33  loss=0.229061
epoch 34  loss=0.228648
epoch 35  loss=0.223993
epoch 36  loss=0.224029
epoch 37  l

In [2]:
# For testing inferrence

# Sample a new 8-bar pattern (T is now inferred from loaded data)
# h, v = sample_pattern(
#     model,
#     T=T,
#     n_iters=16,
#     device="cpu",
#     density_bias=0.0,      # + => busier, - => sparser, range of [-3, 3]
#     sigma_floor=0.08,      # higher => more velocity variation
#     resample_fraction=0.45,
#     temperature_hit=1.0,
# )

# print("hit:", h.numpy().astype(int).tolist())
# midi_vel = (v.numpy() * 127.0).round().astype(int).tolist()
# print("vel:", midi_vel)

# out_path = export_single_voice_steps_to_midi(
#     hit=h.numpy(),
#     vel=v.numpy(),
#     out_dir="./generated_midis",
#     cfg=cfg_out,
# )

# print("Saved MIDI:", out_path)

In [3]:
# Inspect a midi file

# import mido
# from collections import Counter

# def inspect_midi(path: str, max_print: int = 10):
#     mid = mido.MidiFile(path)
#     print("ticks_per_beat:", mid.ticks_per_beat)

#     pitch_counts = Counter()
#     chan_counts = Counter()
#     events = 0

#     for tr_i, track in enumerate(mid.tracks):
#         abs_tick = 0
#         for msg in track:
#             abs_tick += msg.time
#             if msg.type == "note_on" and msg.velocity > 0:
#                 events += 1
#                 chan = getattr(msg, "channel", None)
#                 pitch_counts[msg.note] += 1
#                 chan_counts[chan] += 1

#     print("note_on events:", events)
#     print("top pitches:", pitch_counts.most_common(max_print))
#     print("channels:", chan_counts.most_common())

# inspect_midi("MIDI/1hit.mid")